# Object-Oriented Programming in Python

A complete, hands-on guide to OOP in Python — from the basics to advanced concepts.

## Table of Contents

1. [Introduction to OOP](#1.-Introduction-to-OOP)
2. [Classes and Objects](#2.-Classes-and-Objects)
3. [The `__init__` Method and `self`](#3.-The-__init__-Method-and-self)
4. [Instance vs. Class Attributes](#4.-Instance-vs.-Class-Attributes)
5. [Instance, Class, and Static Methods](#5.-Instance,-Class,-and-Static-Methods)
6. [Encapsulation](#6.-Encapsulation)
7. [Properties (Getters/Setters with `@property`)](#7.-Properties-(Getters/Setters-with-@property))
8. [Inheritance](#8.-Inheritance)
9. [Multiple Inheritance and MRO](#9.-Multiple-Inheritance-and-MRO)
10. [Polymorphism](#10.-Polymorphism)
11. [Abstraction (Abstract Base Classes)](#11.-Abstraction-(Abstract-Base-Classes))
12. [Magic / Dunder Methods](#12.-Magic-/-Dunder-Methods)
13. [Operator Overloading](#13.-Operator-Overloading)
14. [Composition vs. Inheritance](#14.-Composition-vs.-Inheritance)
15. [Custom Exceptions with OOP](#15.-Custom-Exceptions-with-OOP)
16. [Dataclasses](#16.-Dataclasses)
17. [`__slots__` for Memory Optimization](#17.-__slots__-for-Memory-Optimization)
18. [Practice Exercises](#18.-Practice-Exercises)
19. [Mini-Project: Library Management System](#19.-Mini-Project:-Library-Management-System)
20. [Summary Cheat Sheet](#20.-Summary-Cheat-Sheet)


## 1. Introduction to OOP

**Object-Oriented Programming (OOP)** is a programming paradigm based on the concept of *objects*, which bundle together **data** (attributes) and **behavior** (methods).

The four pillars of OOP are:

| Pillar | Description |
|---|---|
| **Encapsulation** | Bundling data and methods together, and restricting direct access to some components |
| **Abstraction** | Hiding complex implementation details and exposing only the essentials |
| **Inheritance** | Creating new classes based on existing ones, reusing code |
| **Polymorphism** | Allowing objects of different classes to be treated through a common interface |

Everything in Python is an object — even integers, strings, and functions!


In [1]:
# Everything in Python is an object
print(type(42))
print(type("hello"))
print(type([1, 2, 3]))
print(type(print))


<class 'int'>
<class 'str'>
<class 'list'>
<class 'builtin_function_or_method'>


## 2. Classes and Objects

- A **class** is a blueprint for creating objects.
- An **object** (or **instance**) is a concrete realization of that blueprint.

Use the `class` keyword to define a class, and call the class like a function to create an instance.


In [2]:
class Dog:
    """A simple class representing a dog."""
    species = "Canis familiaris"  # class attribute, shared by all instances

    def bark(self):
        return "Woof!"


# Creating objects (instances) of the Dog class
dog1 = Dog()
dog2 = Dog()

print(dog1.bark())
print(dog1.species, dog2.species)
print(type(dog1))
print(isinstance(dog1, Dog))


Woof!
Canis familiaris Canis familiaris
<class 'Dog'>
True


## 3. The `__init__` Method and `self`

`__init__` is a special *constructor* method that Python calls automatically when a new object is created. It's where you initialize instance attributes.

`self` refers to the specific instance calling the method — it must be the first parameter of every instance method (though it is passed automatically).


In [3]:
class Dog:
    species = "Canis familiaris"

    def __init__(self, name, age):
        self.name = name   # instance attribute
        self.age = age     # instance attribute

    def bark(self):
        return f"{self.name} says Woof!"

    def describe(self):
        return f"{self.name} is {self.age} year(s) old."


dog1 = Dog("Rex", 3)
dog2 = Dog("Bella", 5)

print(dog1.bark())
print(dog2.describe())
print(dog1.name, dog2.name)  # each instance has its own data


Rex says Woof!
Bella is 5 year(s) old.
Rex Bella


## 4. Instance vs. Class Attributes

- **Instance attributes** belong to a specific object (defined with `self.attr` inside methods, usually `__init__`).
- **Class attributes** are shared across *all* instances of the class.

Be careful: if you reassign a class attribute through an instance, Python creates a new *instance* attribute that shadows the class attribute instead of modifying the shared one.


In [4]:
class Counter:
    total_created = 0  # class attribute — shared counter

    def __init__(self, name):
        self.name = name          # instance attribute
        Counter.total_created += 1  # modify via the class, not the instance


c1 = Counter("A")
c2 = Counter("B")
c3 = Counter("C")

print("Total counters created:", Counter.total_created)
print(c1.name, c2.name, c3.name)

# Mutable class attributes are a common pitfall!
class Bad:
    items = []  # shared across ALL instances

    def add(self, item):
        self.items.append(item)  # mutates the SAME list for every instance!

b1 = Bad()
b2 = Bad()
b1.add("x")
print(b2.items)  # ['x'] -- surprising! b2 sees b1's change


Total counters created: 3
A B C
['x']


## 5. Instance, Class, and Static Methods

| Type | Decorator | First argument | Use case |
|---|---|---|---|
| Instance method | *(none)* | `self` | Operates on a specific instance |
| Class method | `@classmethod` | `cls` | Operates on the class itself; often used as alternate constructors |
| Static method | `@staticmethod` | *(none)* | Utility function related to the class but needing no instance/class data |


In [5]:
class Pizza:
    def __init__(self, toppings):
        self.toppings = toppings

    def __repr__(self):
        return f"Pizza({self.toppings!r})"

    # Alternate constructor using classmethod
    @classmethod
    def margherita(cls):
        return cls(["mozzarella", "tomato"])

    @classmethod
    def pepperoni(cls):
        return cls(["mozzarella", "pepperoni"])

    # Utility function, doesn't need self or cls
    @staticmethod
    def is_valid_topping(topping):
        allowed = {"mozzarella", "tomato", "pepperoni", "basil", "mushroom"}
        return topping in allowed


p1 = Pizza.margherita()
p2 = Pizza.pepperoni()
print(p1, p2)

print(Pizza.is_valid_topping("basil"))
print(Pizza.is_valid_topping("pineapple"))


Pizza(['mozzarella', 'tomato']) Pizza(['mozzarella', 'pepperoni'])
True
False


## 6. Encapsulation

Encapsulation restricts direct access to an object's internals. Python uses **naming conventions** rather than strict access modifiers:

| Prefix | Convention | Meaning |
|---|---|---|
| `name` | public | Freely accessible |
| `_name` | protected (convention only) | "Internal use" — please don't touch from outside |
| `__name` | private (name-mangled) | Python renames it to `_ClassName__name` to avoid accidental access/collisions |


In [6]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner          # public
        self._pin = "1234"          # protected (convention)
        self.__balance = balance    # private (name-mangled)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount > self.__balance:
            raise ValueError("Insufficient funds")
        self.__balance -= amount

    def get_balance(self):
        return self.__balance


acc = BankAccount("Alice", 100)
acc.deposit(50)
acc.withdraw(30)
print(acc.get_balance())  # 120

try:
    print(acc.__balance)  # AttributeError -- not directly accessible
except AttributeError as e:
    print("Error:", e)

# It's technically still reachable via name mangling (not recommended!):
print(acc._BankAccount__balance)


120
Error: 'BankAccount' object has no attribute '__balance'
120


## 7. Properties (Getters/Setters with `@property`)

The `@property` decorator lets you define methods that behave like attributes, enabling validation logic while keeping a clean, attribute-style API.


In [7]:
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius

    @property
    def celsius(self):
        """Getter for celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """Setter with validation."""
        if value < -273.15:
            raise ValueError("Temperature below absolute zero is not possible")
        self._celsius = value

    @property
    def fahrenheit(self):
        """Computed / read-only property."""
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.celsius, t.fahrenheit)

t.celsius = 100
print(t.fahrenheit)

try:
    t.celsius = -300  # triggers validation in the setter
except ValueError as e:
    print("Error:", e)


25 77.0
212.0
Error: Temperature below absolute zero is not possible


## 8. Inheritance

Inheritance allows a class (**subclass/child**) to reuse and extend the behavior of another class (**superclass/parent**). Use `super()` to call the parent class's methods.


In [8]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        raise NotImplementedError("Subclasses must implement this method")

    def __str__(self):
        return f"{self.__class__.__name__}({self.name})"


class Dog(Animal):
    def speak(self):
        return f"{self.name} says Woof!"


class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"


class Puppy(Dog):
    def __init__(self, name, age):
        super().__init__(name)  # reuse the parent's __init__
        self.age = age

    def speak(self):
        parent_sound = super().speak()  # reuse and extend parent behavior
        return f"{parent_sound} (but it's just a tiny yip, it's a puppy)"


animals = [Dog("Rex"), Cat("Whiskers"), Puppy("Buddy", 1)]
for a in animals:
    print(a, "->", a.speak())

print(isinstance(animals[2], Dog))    # True
print(isinstance(animals[2], Animal))  # True
print(issubclass(Puppy, Animal))       # True


Dog(Rex) -> Rex says Woof!
Cat(Whiskers) -> Whiskers says Meow!
Puppy(Buddy) -> Buddy says Woof! (but it's just a tiny yip, it's a puppy)
True
True
True


## 9. Multiple Inheritance and MRO

Python supports **multiple inheritance**: a class can inherit from more than one parent. The **Method Resolution Order (MRO)** determines the order in which base classes are searched when looking up a method. Python uses the **C3 linearization algorithm**.


In [9]:
class Flyer:
    def move(self):
        return "flying"


class Swimmer:
    def move(self):
        return "swimming"


class Duck(Flyer, Swimmer):
    pass


d = Duck()
print(d.move())          # "flying" -- Flyer comes first in the MRO
print(Duck.__mro__)      # Method Resolution Order

# Mixins are a common, clean use of multiple inheritance
class JSONMixin:
    def to_json(self):
        import json
        return json.dumps(self.__dict__)


class Point(JSONMixin):
    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(1, 2)
print(p.to_json())


flying
(<class 'Duck'>, <class 'Flyer'>, <class 'Swimmer'>, <class 'object'>)
{"x": 1, "y": 2}


## 10. Polymorphism

Polymorphism ("many forms") lets objects of different classes be used interchangeably through a shared interface — e.g. calling `.speak()` or `.area()` without knowing (or caring) the object's exact type. Python's *duck typing* ("if it walks like a duck and quacks like a duck...") makes this especially natural.


In [10]:
class Shape:
    def area(self):
        raise NotImplementedError


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


class Triangle(Shape):
    def __init__(self, base, height):
        self.base, self.height = base, height

    def area(self):
        return 0.5 * self.base * self.height


shapes = [Rectangle(3, 4), Circle(5), Triangle(6, 2)]

# Polymorphism: same interface (.area()), different implementations
for shape in shapes:
    print(f"{shape.__class__.__name__} area: {shape.area():.2f}")

total_area = sum(shape.area() for shape in shapes)
print("Total area:", round(total_area, 2))


Rectangle area: 12.00
Circle area: 78.54
Triangle area: 6.00
Total area: 96.54


## 11. Abstraction (Abstract Base Classes)

The `abc` module lets you define **abstract base classes (ABCs)** — classes that cannot be instantiated directly and force subclasses to implement specific methods. This enforces a consistent interface across subclasses.


In [11]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        """Process a payment. Must be implemented by subclasses."""
        ...

    def receipt(self, amount):
        # concrete method, shared by all subclasses
        return f"Charged ${amount:.2f} via {self.__class__.__name__}"


class CreditCard(PaymentMethod):
    def pay(self, amount):
        return f"Paid ${amount:.2f} with credit card"


class PayPal(PaymentMethod):
    def pay(self, amount):
        return f"Paid ${amount:.2f} with PayPal"


try:
    pm = PaymentMethod()  # TypeError: can't instantiate abstract class
except TypeError as e:
    print("Error:", e)

for method in (CreditCard(), PayPal()):
    print(method.pay(49.99))
    print(method.receipt(49.99))


Error: Can't instantiate abstract class PaymentMethod with abstract method pay
Paid $49.99 with credit card
Charged $49.99 via CreditCard
Paid $49.99 with PayPal
Charged $49.99 via PayPal


## 12. Magic / Dunder Methods

"Dunder" (double underscore) methods let your objects integrate with Python's built-in syntax and functions (`print()`, `len()`, `==`, `+`, iteration, context managers, etc.).

Common dunder methods:

| Method | Triggered by |
|---|---|
| `__init__` | Object creation |
| `__repr__` | `repr(obj)` — unambiguous, developer-facing |
| `__str__` | `str(obj)` / `print(obj)` — readable, user-facing |
| `__len__` | `len(obj)` |
| `__eq__` | `obj1 == obj2` |
| `__lt__`, `__le__`, ... | `<`, `<=`, etc. |
| `__add__`, `__sub__`, ... | `+`, `-`, etc. |
| `__getitem__` | `obj[key]` |
| `__iter__` / `__next__` | `for x in obj` |
| `__call__` | `obj(...)` |
| `__enter__` / `__exit__` | `with obj:` |


In [12]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __eq__(self, other):
        return isinstance(other, Vector) and self.x == other.x and self.y == other.y

    def __len__(self):
        return int((self.x ** 2 + self.y ** 2) ** 0.5)

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __getitem__(self, index):
        return (self.x, self.y)[index]


v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(repr(v1))     # uses __repr__
print(str(v1))       # uses __str__
print(v1 == Vector(1, 2))  # uses __eq__
print(len(v2))        # uses __len__ -> magnitude, truncated
print(v1 + v2)        # uses __add__
print(v1[0], v1[1])   # uses __getitem__


Vector(1, 2)
(1, 2)
True
5
(4, 6)
1 2


In [13]:
# __iter__ / __next__ -- making a custom iterable, and __call__

class CountUp:
    """An iterator that counts up to `limit`."""
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration
        self.current += 1
        return self.current


for number in CountUp(5):
    print(number, end=" ")
print()


class Multiplier:
    """Callable object: acts like a function via __call__."""
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor


double = Multiplier(2)
print(double(21))  # calling the OBJECT like a function


1 2 3 4 5 
42


## 13. Operator Overloading

By implementing dunder methods, you can make custom objects work naturally with Python's built-in operators — a specific application of the magic methods above.


In [14]:
class Money:
    def __init__(self, amount, currency="USD"):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f"{self.amount:.2f} {self.currency}"

    def _check_currency(self, other):
        if self.currency != other.currency:
            raise ValueError("Cannot operate on different currencies")

    def __add__(self, other):
        self._check_currency(other)
        return Money(self.amount + other.amount, self.currency)

    def __sub__(self, other):
        self._check_currency(other)
        return Money(self.amount - other.amount, self.currency)

    def __mul__(self, factor):
        return Money(self.amount * factor, self.currency)

    def __lt__(self, other):
        self._check_currency(other)
        return self.amount < other.amount

    def __le__(self, other):
        self._check_currency(other)
        return self.amount <= other.amount


wallet = Money(50) + Money(25) - Money(10)
print(wallet)             # 65.00 USD
print(wallet * 2)          # 130.00 USD
print(Money(10) < Money(20))  # True
print(sorted([Money(30), Money(5), Money(15)], key=lambda m: m.amount))


65.00 USD
130.00 USD
True
[5.00 USD, 15.00 USD, 30.00 USD]


## 14. Composition vs. Inheritance

- **Inheritance** models an **"is-a"** relationship (a `Puppy` *is a* `Dog`).
- **Composition** models a **"has-a"** relationship (a `Car` *has an* `Engine`).

A common design guideline: *"favor composition over inheritance"* — it tends to produce more flexible, loosely-coupled designs.


In [15]:
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        return f"Engine ({self.horsepower} HP) starting..."


class GPS:
    def navigate(self, destination):
        return f"Navigating to {destination}..."


class Car:
    """A Car HAS-A Engine and HAS-A GPS (composition), rather than inheriting from them."""
    def __init__(self, make, model, horsepower):
        self.make = make
        self.model = model
        self.engine = Engine(horsepower)  # composition
        self.gps = GPS()                  # composition

    def start(self):
        return f"{self.make} {self.model}: {self.engine.start()}"

    def navigate(self, destination):
        return self.gps.navigate(destination)


car = Car("Toyota", "Corolla", 140)
print(car.start())
print(car.navigate("Downtown"))


Toyota Corolla: Engine (140 HP) starting...
Navigating to Downtown...


## 15. Custom Exceptions with OOP

Exceptions in Python are classes too! You can create your own exception hierarchy by subclassing `Exception` (or a more specific built-in exception).


In [16]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance."""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(
            f"Cannot withdraw ${amount:.2f}: balance is only ${balance:.2f}"
        )


class NegativeAmountError(ValueError):
    """Raised when a negative amount is supplied."""
    pass


class Account:
    def __init__(self, balance=0):
        self.balance = balance

    def withdraw(self, amount):
        if amount < 0:
            raise NegativeAmountError("Amount cannot be negative")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return self.balance


acc = Account(100)
try:
    acc.withdraw(500)
except InsufficientFundsError as e:
    print("Caught:", e)

try:
    acc.withdraw(-10)
except NegativeAmountError as e:
    print("Caught:", e)
except InsufficientFundsError as e:
    print("Caught:", e)


Caught: Cannot withdraw $500.00: balance is only $100.00
Caught: Amount cannot be negative


## 16. Dataclasses

The `dataclasses` module (Python 3.7+) auto-generates boilerplate like `__init__`, `__repr__`, and `__eq__` for classes that primarily hold data.


In [17]:
from dataclasses import dataclass, field


@dataclass
class Point:
    x: float
    y: float

    def distance_from_origin(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5


p1 = Point(3, 4)
p2 = Point(3, 4)
print(p1)                 # auto-generated __repr__
print(p1 == p2)            # auto-generated __eq__ (compares field values)
print(p1.distance_from_origin())


@dataclass(order=True)
class Student:
    name: str
    grade: float
    tags: list = field(default_factory=list)  # safe mutable default


s1 = Student("Alice", 92.5, ["honors"])
s2 = Student("Bob", 88.0)
print(s1, s2)
print(sorted([s1, s2]))  # order=True enables comparisons, field order matters


Point(x=3, y=4)
True
5.0
Student(name='Alice', grade=92.5, tags=['honors']) Student(name='Bob', grade=88.0, tags=[])
[Student(name='Alice', grade=92.5, tags=['honors']), Student(name='Bob', grade=88.0, tags=[])]


## 17. `__slots__` for Memory Optimization

By default, instances store their attributes in a per-instance `__dict__`, which is flexible but memory-heavy. Defining `__slots__` tells Python to allocate fixed storage for a known set of attributes, saving memory (useful when creating many instances) — at the cost of no longer being able to add arbitrary new attributes dynamically.


In [18]:
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class SlottedPoint:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


rp = RegularPoint(1, 2)
sp = SlottedPoint(1, 2)

rp.z = 99  # OK -- __dict__ allows arbitrary new attributes
print(rp.z)

try:
    sp.z = 99  # AttributeError -- no __dict__, only x and y are allowed
except AttributeError as e:
    print("Error:", e)

import sys
print("Regular instance dict:", rp.__dict__)
# print("Slotted has no __dict__:", sp.__dict__)  # would raise AttributeError


99
Error: 'SlottedPoint' object has no attribute 'z'
Regular instance dict: {'x': 1, 'y': 2, 'z': 99}


## 18. Practice Exercises

Try these on your own before checking the (hidden) solutions further below!

1. **Employee hierarchy**: Create an `Employee` base class with `name` and `salary`. Create `Manager` and `Developer` subclasses that override a `describe()` method. Managers should also track a list of reports (composition).
2. **Shape polymorphism**: Extend the `Shape` example from Section 10 with a `Square` class and a `perimeter()` method for every shape.
3. **Custom stack**: Implement a `Stack` class using `__len__`, `__getitem__`, `push()`, and `pop()`, so it behaves like a list-ish container.
4. **Validated property**: Create a `Person` class with a `age` property that raises `ValueError` if set to a negative number.
5. **Abstract interface**: Define an abstract `Notifier` class with an abstract `send(message)` method, then implement `EmailNotifier` and `SMSNotifier` subclasses.


In [19]:
# Space for you to try the exercises above!




### Example Solutions

Expand and run the cell below to compare with your own attempts.


In [20]:
# --- Solution 1: Employee hierarchy ---
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def describe(self):
        return f"{self.name} earns ${self.salary:,.2f}"


class Developer(Employee):
    def __init__(self, name, salary, language):
        super().__init__(name, salary)
        self.language = language

    def describe(self):
        return f"{super().describe()} and codes in {self.language}"


class Manager(Employee):
    def __init__(self, name, salary):
        super().__init__(name, salary)
        self.reports = []  # composition: a Manager HAS employees

    def add_report(self, employee):
        self.reports.append(employee)

    def describe(self):
        names = ", ".join(r.name for r in self.reports) or "nobody"
        return f"{super().describe()} and manages {names}"


dev = Developer("Sam", 90000, "Python")
mgr = Manager("Jordan", 120000)
mgr.add_report(dev)
print(dev.describe())
print(mgr.describe())


# --- Solution 3: Custom Stack ---
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def __len__(self):
        return len(self._items)

    def __getitem__(self, index):
        return self._items[index]

    def __repr__(self):
        return f"Stack({self._items})"


s = Stack()
s.push(1)
s.push(2)
s.push(3)
print(s, len(s), s[0], s.pop())
print(s)


Sam earns $90,000.00 and codes in Python
Jordan earns $120,000.00 and manages Sam
Stack([1, 2]) 3 1 3
Stack([1, 2])


## 19. Mini-Project: Library Management System

A small end-to-end example that ties together inheritance, encapsulation, composition, polymorphism, abstraction, and dunder methods.


In [21]:
from abc import ABC, abstractmethod
from datetime import date


class LibraryItem(ABC):
    """Abstract base for anything the library lends out."""

    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self._checked_out = False

    @abstractmethod
    def loan_period_days(self):
        """Number of days this item type can be borrowed for."""
        ...

    @property
    def is_available(self):
        return not self._checked_out

    def checkout(self):
        if self._checked_out:
            raise ValueError(f"'{self.title}' is already checked out")
        self._checked_out = True

    def return_item(self):
        self._checked_out = False

    def __repr__(self):
        status = "available" if self.is_available else "checked out"
        return f"<{self.__class__.__name__} {self.item_id!r} '{self.title}' ({status})>"


class Book(LibraryItem):
    def __init__(self, title, item_id, author):
        super().__init__(title, item_id)
        self.author = author

    def loan_period_days(self):
        return 21


class DVD(LibraryItem):
    def loan_period_days(self):
        return 7


class Library:
    """Composition: a Library HAS-A collection of LibraryItems and tracks members."""

    def __init__(self, name):
        self.name = name
        self._catalog = {}   # item_id -> LibraryItem
        self._loans = {}     # item_id -> (member_name, due_date placeholder)

    def add_item(self, item: LibraryItem):
        self._catalog[item.item_id] = item

    def checkout(self, item_id, member_name):
        item = self._catalog[item_id]
        item.checkout()  # polymorphic: works for Book, DVD, or any future subclass
        self._loans[item_id] = member_name
        print(f"{member_name} borrowed '{item.title}' for {item.loan_period_days()} days.")

    def return_item(self, item_id):
        item = self._catalog[item_id]
        item.return_item()
        self._loans.pop(item_id, None)
        print(f"'{item.title}' has been returned.")

    def available_items(self):
        return [item for item in self._catalog.values() if item.is_available]

    def __len__(self):
        return len(self._catalog)

    def __iter__(self):
        return iter(self._catalog.values())


# --- Demo ---
library = Library("City Library")
library.add_item(Book("Dune", "B001", "Frank Herbert"))
library.add_item(Book("Foundation", "B002", "Isaac Asimov"))
library.add_item(DVD("Interstellar", "D001"))

print(f"Catalog has {len(library)} items:")
for item in library:
    print(" ", item)

library.checkout("B001", "Alice")
library.checkout("D001", "Bob")

print("\nAvailable items now:")
for item in library.available_items():
    print(" ", item)

library.return_item("B001")
print("\nAfter returning Dune:")
for item in library:
    print(" ", item)

try:
    library.checkout("D001", "Charlie")  # already checked out by Bob
except ValueError as e:
    print("\nError:", e)


Catalog has 3 items:
  <Book 'B001' 'Dune' (available)>
  <Book 'B002' 'Foundation' (available)>
  <DVD 'D001' 'Interstellar' (available)>
Alice borrowed 'Dune' for 21 days.
Bob borrowed 'Interstellar' for 7 days.

Available items now:
  <Book 'B002' 'Foundation' (available)>
'Dune' has been returned.

After returning Dune:
  <Book 'B001' 'Dune' (available)>
  <Book 'B002' 'Foundation' (available)>
  <DVD 'D001' 'Interstellar' (checked out)>

Error: 'Interstellar' is already checked out


## 20. Summary Cheat Sheet

| Concept | Syntax / Keyword |
|---|---|
| Define a class | `class Foo:` |
| Constructor | `def __init__(self, ...):` |
| Instance attribute | `self.attr = value` |
| Class attribute | defined directly in the class body |
| Instance method | `def method(self, ...):` |
| Class method | `@classmethod` + `def method(cls, ...):` |
| Static method | `@staticmethod` + `def method(...):` |
| Inheritance | `class Sub(Base):` |
| Call parent method | `super().method(...)` |
| Abstract class | `class Foo(ABC):` + `@abstractmethod` |
| Property | `@property` / `@x.setter` |
| Private attribute | `self.__attr` (name-mangled) |
| Protected attribute | `self._attr` (convention) |
| String representation | `__repr__`, `__str__` |
| Equality | `__eq__` |
| Operator overload | `__add__`, `__sub__`, `__mul__`, ... |
| Container behavior | `__len__`, `__getitem__`, `__iter__` |
| Callable object | `__call__` |
| Auto-generated class | `@dataclass` |
| Memory-efficient class | `__slots__ = (...)` |

### Key takeaways

- Classes bundle **data + behavior**; objects are instances of classes.
- Use **encapsulation** to protect internal state, exposing a clean API via properties.
- Use **inheritance** for "is-a" relationships and **composition** for "has-a" relationships — prefer composition when in doubt.
- **Polymorphism** and **abstraction** let you write code that works with many types through a shared interface.
- **Dunder methods** are how you make your custom objects feel like native Python objects.
- Modern Python offers shortcuts like `@dataclass` to reduce boilerplate.

Happy coding! 🐍
